In [ ]:
%pip install python-dotenv

In [ ]:
import os
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph


class AgentState(TypedDict):
    topic:str
    article:str


llm = ChatOllama(model="llama3.2:1b")

def write_node(state:AgentState):
    """Single content writer — no tools, no collaboration."""
    topic = state["topic"]

    messages = [
        SystemMessage(content=(
             "You are a seasoned Content Writer. "
            "Your goal is to craft a well-structured, publication-ready article."
        )),
        HumanMessage(content=f"Write a detailed article about: {topic}")
    ]
    response = llm.invoke(messages)
    return {"article": response.content}

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langgraph.graph import END, START


graph = StateGraph(AgentState)
graph.add_node("writer", write_node)
graph.set_entry_point("writer")
graph.add_edge("writer", END)

app = graph.compile()

# ── 5. Run ───────────────────────────────────────────────────────
TOPIC = "The Future of Space Exploration"
result = app.invoke({"topic": TOPIC, "article": ""})
print(result["article"])

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent


search_tool = TavilySearchResults(max_results=3)

class WriterState(TypedDict):
    topic:str
    article:str

WRITER_SYSTEM_PROMPT = """You are a Content Writer with access to a web search tool.
Before writing, always search the web to gather current, accurate facts about the topic.
Your article must reference up-to-date information with specific details, dates, and sources.
"""

writer_agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt=WRITER_SYSTEM_PROMPT,
)

def writer_with_search_node(state:WriterState) -> WriterState:
    topic = state["topic"]
    result = writer_agent.invoke({
        "messages": [HumanMessage(content=f"Write a detailed article about: {topic}")]
    })

    final_message = result["messages"][-1]
    return {"article": final_message.content}


graph = StateGraph(WriterState)
graph.add_node("writer", writer_with_search_node)
graph.set_entry_point("writer")
graph.add_edge("writer", END)

app = graph.compile()


result = app.invoke({"topic": TOPIC, "article": ""})
print(result["article"])

In [ ]:
class AgentState(TypedDict):
    topic: str
    research_outline: str   # produced by Researcher
    draft_article: str      # produced by Writer
    final_article: str      # produced by Editor


researcher_agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt=(
        "You are a Content Researcher. Use the web search tool to gather "
        "fresh, accurate information about the given topic. "
        "Produce a structured outline with key findings, statistics, "
        "and section headings for a writer to expand upon."
    )
)

writer_agent = create_agent(
    model=llm,
    system_prompt=(
        "You are a Content Writer. You receive a research outline and "
        "expand it into a full, well-structured article. "
        "Use all the provided facts and figures. "
        "Write in a clear, engaging style suitable for a general audience."
    )
)

# Agent 3: Content Editor (no tools — polishes the draft)
editor_agent = create_agent(
    model=llm,
    system_prompt=(
        "You are a Content Editor. You receive a draft article and "
        "improve its clarity, grammar, flow, and structure. "
        "Ensure it reads like a polished, publication-ready piece."
    )
)

def researcher_node(state: AgentState) -> AgentState:
    topic = state["topic"]
    result = researcher_agent.invoke({
        "messages": [HumanMessage(content=(
            f"Research the topic '{topic}' and produce a detailed outline "
            f"with key findings, data points, and section headings."
        ))]
    })
    return {"research_outline": result["messages"][-1].content}


def writer_node(state: AgentState) -> AgentState:
    outline = state["research_outline"]
    result = writer_agent.invoke({
        "messages": [HumanMessage(content=(
            f"Using this research outline, write a full article:\n\n{outline}"
        ))]
    })
    return {"draft_article": result["messages"][-1].content}


def editor_node(state: AgentState) -> AgentState:
    draft = state["draft_article"]
    result = editor_agent.invoke({
        "messages": [HumanMessage(content=(
            f"Edit and polish this article draft:\n\n{draft}"
        ))]
    })
    return {"final_article": result["messages"][-1].content}


graph = StateGraph(AgentState)

graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("editor", editor_node)

graph.set_entry_point("researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "editor")
graph.add_edge("editor", END)

app = graph.compile()

# ── 5. Run ───────────────────────────────────────────────────────
result = app.invoke({
    "topic": TOPIC,
    "research_outline": "",
    "draft_article": "",
    "final_article": ""
})

print("=== RESEARCH OUTLINE ===")
print(result["research_outline"])
print("\n=== DRAFT ===")
print(result["draft_article"])
print("\n=== FINAL ARTICLE ===")
print(result["final_article"])

In [ ]:
import os
from typing import TypedDict, Optional, List
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END
from langchain.agents import create_agent
from pydantic import BaseModel, Field

load_dotenv()

# ─── LLM & Tools ─────────────────────────────────────────────────
llm         = ChatOllama(model="llama3.2:1b", temperature=0.5)
search_tool = TavilySearchResults(max_results=3)

# ─── Pydantic Schemas ─────────────────────────────────────────────
class OutlineSection(BaseModel):
    heading: str      = Field(description="Section heading, max 10 words")
    bullet_points: List[str] = Field(description="3–5 key points for this section")
    sources: List[str]       = Field(description="URLs or references used")

class ResearchOutline(BaseModel):
    article_title: str        = Field(description="Compelling title, max 15 words")
    introduction_hook: str    = Field(description="1–2 sentence opening hook with a specific fact")
    sections: List[OutlineSection] = Field(description="4–5 structured sections")
    key_takeaway: str         = Field(description="Single most important insight")

class ArticleSection(BaseModel):
    heading: str  = Field(description="H2 heading for this section")
    content: str  = Field(description="150–200 word section content with inline Markdown links")

class Article(BaseModel):
    article_title: str             = Field(description="Final article title")
    introduction: str              = Field(description="80–100 word opening paragraph")
    sections: List[ArticleSection] = Field(description="4–5 body sections")
    conclusion: str                = Field(description="80–100 word forward-looking conclusion")
    word_count_estimate: int       = Field(description="Estimated total word count")

# ─── Shared State ─────────────────────────────────────────────────
class AgentState(TypedDict):
    topic: str
    research_outline: Optional[ResearchOutline]
    draft_article: Optional[Article]
    final_article: Optional[Article]

# ─── Agent Definitions ────────────────────────────────────────────
RESEARCHER_SYSTEM = """You are a Senior Research Analyst.
Use the web search tool to find 3–5 credible sources from 2023–2025.
Identify key themes, statistics, mission names, dates, and company names.
Be thorough and specific — your outline directly shapes the article quality.
"""

WRITER_SYSTEM = """You are a Science and Technology Journalist.
Write 700–900 words. Structure: Title → Introduction → 4 body sections → Conclusion.
Tone: Analytical yet accessible (Scientific American style).
Use inline Markdown hyperlinks for all cited sources.
"""

EDITOR_SYSTEM = """You are a Chief Editor.
Review the draft for: factual consistency, tight writing, compelling opening,
correct Markdown hyperlinks, and a substantive conclusion that synthesizes (not repeats).
Return only the polished article.
"""

researcher_base  = create_agent(model=llm, tools=[search_tool], prompt=RESEARCHER_SYSTEM)
structured_res   = llm.with_structured_output(ResearchOutline)
structured_write = llm.with_structured_output(Article)
structured_edit  = llm.with_structured_output(Article)

# ─── Node Functions ───────────────────────────────────────────────
def researcher_node(state: AgentState) -> AgentState:
    raw = researcher_base.invoke({
        "messages": [HumanMessage(content=(
            f"Research: {state['topic']}. Return detailed findings with URLs."
        ))]
    })
    outline = structured_res.invoke(
        f"Create a structured outline from this research:\n\n{raw['messages'][-1].content}"
    )
    return {"research_outline": outline}

def writer_node(state: AgentState) -> AgentState:
    draft = structured_write.invoke(
        f"Write a full article from this outline:\n\n"
        f"{state['research_outline'].model_dump_json(indent=2)}"
    )
    return {"draft_article": draft}

def editor_node(state: AgentState) -> AgentState:
    final = structured_edit.invoke(
        f"Edit and polish this article:\n\n"
        f"{state['draft_article'].model_dump_json(indent=2)}"
    )
    return {"final_article": final}

# ─── Build the Graph ──────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("researcher", researcher_node)
graph.add_node("writer",     writer_node)
graph.add_node("editor",     editor_node)
graph.set_entry_point("researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer",     "editor")
graph.add_edge("editor",     END)

app = graph.compile()

# ─── Run ──────────────────────────────────────────────────────────
TOPIC = "The Future of Space Exploration"

result = app.invoke({
    "topic": TOPIC,
    "research_outline": None,
    "draft_article": None,
    "final_article": None
})

# ─── Display ──────────────────────────────────────────────────────
def display_article(article: Article):
    print(f"# {article.article_title}\n")
    print(f"{article.introduction}\n")
    for section in article.sections:
        print(f"## {section.heading}\n{section.content}\n")
    print(f"---\n{article.conclusion}")
    print(f"\n*(~{article.word_count_estimate} words)*")

display_article(result["final_article"])

In [ ]:
%pip install firecrawl-py

In [12]:
import os
import requests
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class ScrapeInput(BaseModel):
    url: str = Field(description="The full URL of the web page to scrape")



@tool(args_schema=ScrapeInput)
def scrape_website(url:str) -> str:
    """
    Scrapes a web page using the Firecrawl API and returns its full content
    in Markdown format. Use this after finding relevant URLs with the search
    tool to extract the complete text, not just the snippet.
    """
    api_key = os.getenv("FIRECRAWL_API_KEY")

    try:
        response = requests.post(
            "https://api.firecrawl.dev/v1/scrape",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "url": url,
                "formats": ["markdown"],
                "onlyMainContent": True   # strips nav, footer, ads
            },
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        if data.get("success"):
            content = data["data"].get("markdown", "")
            # Truncate to avoid blowing up the context window
            return content[:8000] if len(content) > 8000 else content
        else:
            return f"Scrape failed: {data.get('error', 'Unknown error')}"

    except requests.exceptions.Timeout:
        return "Error: Request timed out. The page may be too slow to respond."
    except requests.exceptions.RequestException as e:
        return f"Error scraping {url}: {str(e)}"


In [21]:
structured_researcher_llm = llm.with_structured_output(ResearchOutline)
structured_writer_llm     = llm.with_structured_output(Article)
structured_editor_llm     = llm.with_structured_output(Article)

In [17]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent

search_tool  = TavilySearchResults(max_results=3)

RESEARCHER_SYSTEM = """You are a Senior Research Analyst.

Your research process MUST follow these steps in order:
1. Use the search tool to find 3–5 relevant, recent URLs about the topic.
2. Pick the 2 most authoritative URLs from the results.
3. Use the scrape tool on each URL to read the full page content.
4. Synthesize what you've read into a structured outline.

Rules:
- Prefer official sources: NASA, ESA, Nature, peer-reviewed journals, reputable news.
- Extract specific facts: mission names, dates, statistics, quotes, company names.
- Your outline is the foundation the Writer will build on — be specific, not vague.
"""

researcher_agent = create_agent(
    model=llm,
    tools=[search_tool, scrape_website],   # ← both tools
    system_prompt=RESEARCHER_SYSTEM
)

In [16]:
def researcher_node(state: AgentState) -> AgentState:
    result = researcher_agent.invoke({
        "messages": [HumanMessage(content=(
            f"Research the topic: '{state['topic']}'\n\n"
            "Step 1: Search for relevant URLs.\n"
            "Step 2: Scrape the top 2 URLs to read their full content.\n"
            "Step 3: Return a detailed structured outline with specific facts."
        ))]
    })

    outline = structured_researcher_llm.invoke(
        f"Convert this research into a structured outline:\n\n"
        f"{result['messages'][-1].content}"
    )
    return {"research_outline": outline}

In [18]:
from typing import Literal
def validate_article_length(state:AgentState) -> Literal["approved", "retry_editor"]:
    """
    Guardrail: checks that the final article word count falls between 500 and 1500.
    Returns 'approved' to proceed, or 'retry_editor' to loop back.
    """

    article = state["final_article"]
    if article is None:
        print("⚠️  Guardrail: No article found. Sending back to editor.")
        return "retry_editor"
        

    total_words = len(article.introduction.split())
    total_words += sum(len(s.content.split()) for s in article.sections)
    total_words += len(article.conclusion.split())

    MIN_WORDS = 500
    MAX_WORDS = 1500    

    if total_words < MIN_WORDS:
        print(f"⚠️  Guardrail FAILED: Article too short ({total_words} words). "
              f"Minimum is {MIN_WORDS}. Retrying editor.")
        # Store the issue in state so the editor can fix it
        state["validation_feedback"] = (
            f"Article is too short ({total_words} words). "
            f"Expand each section to reach at least {MIN_WORDS} words total."
        )
        return "retry_editor"

    if total_words > MAX_WORDS:
        print(f"⚠️  Guardrail FAILED: Article too long ({total_words} words). "
              f"Maximum is {MAX_WORDS}. Retrying editor.")
        state["validation_feedback"] = (
            f"Article is too long ({total_words} words). "
            f"Trim content to stay under {MAX_WORDS} words total."
        )
        return "retry_editor"

    print(f"✅  Guardrail PASSED: Article is {total_words} words. Approved.")
    return "approved"


def validate_article_headings(state: AgentState) -> Literal["approved", "retry_editor"]:
    """Guardrail: ensures the article has at least 4 body sections."""
    article = state.get("final_article")

    if article is None or len(article.sections) < 4:
        section_count = len(article.sections) if article else 0
        print(f"⚠️  Guardrail FAILED: Only {section_count} sections found. Need at least 4.")
        state["validation_feedback"] = (
            f"Article has only {section_count} sections. "
            "Add more sections to cover the topic comprehensively."
        )
        return "retry_editor"

    print(f"✅  Guardrail PASSED: {len(article.sections)} sections found.")
    return "approved"


def validate_section_length(state: AgentState) -> Literal["approved", "retry_editor"]:
    """Guardrail: ensures every section is between 100 and 300 words."""
    article = state.get("final_article")
    if article is None:
        return "retry_editor"

    short_sections = []
    for section in article.sections:
        word_count = len(section.content.split())
        if word_count < 100 or word_count > 300:
            short_sections.append(f"'{section.heading}' ({word_count} words)")

    if short_sections:
        print(f"⚠️  Guardrail FAILED: Sections out of range: {short_sections}")
        state["validation_feedback"] = (
            f"These sections need adjustment: {', '.join(short_sections)}. "
            "Each section must be 100–300 words."
        )
        return "retry_editor"

    print("✅  Guardrail PASSED: All sections within word range.")
    return "approved"

In [19]:
def run_all_guardrails(state:AgentState) -> Literal["approved", "retry_editor"]:
    """Runs all guardrail checks. Returns 'retry_editor' if any check fails."""
    checks = [
        validate_article_length,
        validate_article_headings,
        validate_section_length,
    ]

    for check in checks:
        result = check(state)
        if result == "retry_editor":
            return "retry_editor"
    return "approved"



In [30]:
from typing import Optional

class AgentState(TypedDict):
    topic: str
    research_outline: Optional[ResearchOutline]
    draft_article: Optional[Article]
    final_article: Optional[Article]
    validation_feedback: str          # ← new: carries guardrail error messages
    retry_count: int                  # ← new: prevents infinite loops
    human_outline_feedback: str     # ← new
    human_article_feedback: str     # ← new
    human_approved: bool            # ← new

In [ ]:
structured_editor_llm = llm.with_structured_output(Article)

In [24]:
def guardrail_or_stop(state: AgentState) -> Literal["approved", "retry_editor", "force_end"]:
    """Prevents infinite retry loops by capping retries at MAX_RETRIES."""
    if state.get("retry_count", 0) >= MAX_RETRIES:
        print(f"⚠️  Max retries ({MAX_RETRIES}) reached. Force-ending pipeline.")
        return "force_end"
    return run_all_guardrails(state)


In [ ]:
graph = StateGraph(AgentState)

graph.add_node("researcher", researcher_node)
graph.add_node("writer",     writer_node)
graph.add_node("editor",     editor_node)

graph.set_entry_point("researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer",     "editor")

graph.add_conditional_edges(
    "editor",
    guardrail_or_stop,
    {
        "approved":      END,           # passes all checks → done
        "retry_editor":  "editor",      # fails a check → retry
        "force_end":     END            # too many retries → bail out
    }
)

app = graph.compile()

result = app.invoke({
    "topic": TOPIC,
    "research_outline": None,
    "draft_article": None,
    "final_article": None,
    "validation_feedback": "",
    "retry_count": 0
})

In [27]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [28]:
from langgraph.types import interrupt

def human_review_outline_node(state: AgentState) -> AgentState:
    """
    Pauses execution and presents the research outline to a human reviewer.
    The human can approve it or provide revision instructions.
    """
    outline = state["research_outline"]
    # Display the outline for the reviewer
    print("\n" + "="*60)
    print("HUMAN REVIEW — Research Outline")
    print("="*60)
    print(f"Title: {outline.article_title}")
    print(f"Hook:  {outline.introduction_hook}")
    print(f"\nSections ({len(outline.sections)}):")
    for i, section in enumerate(outline.sections, 1):
        print(f"  {i}. {section.heading}")
        for bullet in section.bullet_points:
            print(f"     • {bullet}")
    print(f"\nKey takeaway: {outline.key_takeaway}")
    print("="*60)

    human_feedback = interrupt(
        "Review the outline above. Type 'approve' to continue, "
        "or describe what needs to change:"
    )

    if human_feedback.strip().lower() == "approve":
        print("✅  Outline approved. Proceeding to writing.")
        return {}   # no state change needed
    else:
        # Store feedback; the researcher will use it on a revision pass
        print(f"📝  Revision requested: {human_feedback}")
        return {"human_outline_feedback": human_feedback}


def human_review_article_node(state: AgentState) -> AgentState:
    """
    Pauses execution after editing and presents the final article for approval.
    """
    article = state["final_article"]

    print("\n" + "="*60)
    print("HUMAN REVIEW — Final Article")
    print("="*60)
    print(f"Title: {article.article_title}")
    print(f"\nIntroduction:\n{article.introduction}")
    for section in article.sections:
        print(f"\n## {section.heading}\n{section.content}")
    print(f"\nConclusion:\n{article.conclusion}")
    print(f"\nEstimated word count: {article.word_count_estimate}")
    print("="*60)

    human_feedback = interrupt(
        "Review the article above. Type 'approve' to publish, "
        "or describe what needs to change:"
    )

    if human_feedback.strip().lower() == "approve":
        print("✅  Article approved. Ready to publish.")
        return {"human_approved": True}
    else:
        print(f"📝  Revision requested: {human_feedback}")
        return {"human_article_feedback": human_feedback, "human_approved": False}

In [29]:
def route_after_outline_review(state: AgentState) -> Literal["writer", "researcher"]:
    """If the human requested changes, send back to researcher. Otherwise proceed."""
    if state.get("human_outline_feedback"):
        return "researcher"   # revise the outline
    return "writer"


def route_after_article_review(state: AgentState) -> Literal["approved", "editor"]:
    """If the human approved, finish. Otherwise send back to editor."""
    if state.get("human_approved"):
        return "approved"
    return "editor"

In [ ]:
import os, json, uuid
from typing import TypedDict, Optional, List, Literal
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from pydantic import BaseModel, Field
import requests

load_dotenv()

# ─── LLM & Tools ──────────────────────────────────────────────────────────────
llm         = ChatOllama(model="llama3.2:1b", temperature=0.7)
search_tool = TavilySearchResults(max_results=3)

class ScrapeInput(BaseModel):
    url: str = Field(description="Full URL of the web page to scrape")

@tool(args_schema=ScrapeInput)
def scrape_webpage(url: str) -> str:
    """Scrapes a web page and returns its full Markdown content."""
    try:
        resp = requests.post(
            "https://api.firecrawl.dev/v1/scrape",
            headers={"Authorization": f"Bearer {os.getenv('FIRECRAWL_API_KEY')}",
                     "Content-Type": "application/json"},
            json={"url": url, "formats": ["markdown"], "onlyMainContent": True},
            timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        content = data["data"].get("markdown", "") if data.get("success") else ""
        return content[:8000]
    except Exception as e:
        return f"Scrape error: {e}"

# ─── Pydantic Schemas ──────────────────────────────────────────────────────────
class OutlineSection(BaseModel):
    heading: str           = Field(description="Section heading, max 10 words")
    bullet_points: List[str] = Field(description="3–5 key points")
    sources: List[str]     = Field(description="URLs or references")

class ResearchOutline(BaseModel):
    article_title: str     = Field(description="Title, max 15 words")
    introduction_hook: str = Field(description="1–2 sentence hook with a specific fact")
    sections: List[OutlineSection] = Field(description="4–5 sections")
    key_takeaway: str      = Field(description="Single most important insight")

class ArticleSection(BaseModel):
    heading: str = Field(description="H2 heading")
    content: str = Field(description="150–200 words with inline Markdown links")

class Article(BaseModel):
    article_title: str             = Field(description="Final title")
    introduction: str              = Field(description="80–100 word opening paragraph")
    sections: List[ArticleSection] = Field(description="4–5 body sections")
    conclusion: str                = Field(description="80–100 word forward-looking conclusion")
    word_count_estimate: int       = Field(description="Estimated total word count")

# ─── Shared State ─────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    topic: str
    raw_research: str
    research_outline: Optional[ResearchOutline]
    draft_article: Optional[Article]
    final_article: Optional[Article]
    validation_feedback: str
    retry_count: int
    human_outline_feedback: str
    human_article_feedback: str
    human_approved: bool
    published_filepath: str

# ─── Agents ───────────────────────────────────────────────────────────────────
researcher_agent = create_agent(
    model=llm, tools=[search_tool, scrape_webpage],
    prompt="Search for relevant URLs, then scrape the top 2 for full content. "
           "Return detailed findings with specific facts, dates, and URLs."
)
structured_researcher_llm = llm.with_structured_output(ResearchOutline)
structured_writer_llm     = llm.with_structured_output(Article)
structured_editor_llm     = llm.with_structured_output(Article)

MAX_RETRIES = 3

# ─── Nodes ────────────────────────────────────────────────────────────────────
def researcher_node(state):
    feedback = state.get("human_outline_feedback", "")
    query = f"Research: {state['topic']}."
    if feedback:
        query += f" Revision: {feedback}"
    raw  = researcher_agent.invoke({"messages": [HumanMessage(content=query)]})
    text = raw["messages"][-1].content
    outline = structured_researcher_llm.invoke(f"Create outline:\n\n{text}")
    return {"raw_research": text, "research_outline": outline, "human_outline_feedback": ""}

def human_review_outline_node(state):
    outline = state["research_outline"]
    print(f"\n⏸  OUTLINE REVIEW: '{outline.article_title}' ({len(outline.sections)} sections)")
    feedback = interrupt("Type 'approve' or describe changes needed:")
    if feedback.strip().lower() == "approve":
        return {}
    return {"human_outline_feedback": feedback}

def writer_node(state):
    prompt = (f"Write article from outline:\n\n{state['research_outline'].model_dump_json(indent=2)}"
              f"\n\nRaw research:\n{state.get('raw_research','')[:3000]}")
    draft = structured_writer_llm.invoke(prompt)
    return {"draft_article": draft}

def editor_node(state):
    prompt = (
        f"Edit this draft:\n{state['draft_article'].model_dump_json(indent=2)}\n\n"
        f"Verify against outline:\n{state['research_outline'].model_dump_json(indent=2)}\n\n"
        f"Raw research:\n{state.get('raw_research','')[:2000]}"
    )
    if state.get("validation_feedback"):
        prompt += f"\n\nFix this: {state['validation_feedback']}"
    if state.get("human_article_feedback"):
        prompt += f"\n\nHuman feedback: {state['human_article_feedback']}"
    final = structured_editor_llm.invoke(prompt)
    return {"final_article": final, "retry_count": state.get("retry_count",0)+1,
            "validation_feedback": "", "human_article_feedback": ""}

def human_review_article_node(state):
    article = state["final_article"]
    print(f"\n⏸  ARTICLE REVIEW: '{article.article_title}' (~{article.word_count_estimate} words)")
    feedback = interrupt("Type 'approve' to publish or describe changes:")
    if feedback.strip().lower() == "approve":
        return {"human_approved": True}
    return {"human_article_feedback": feedback, "human_approved": False}

def publish_node(state):
    if not state.get("human_approved"):
        return {}
    article   = state["final_article"]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath  = f"outputs/article_{timestamp}.md"
    Path("outputs").mkdir(exist_ok=True)
    lines = [f"# {article.article_title}\n\n{article.introduction}\n"]
    for s in article.sections:
        lines.append(f"## {s.heading}\n\n{s.content}\n")
    lines.append(f"---\n{article.conclusion}")
    Path(filepath).write_text("\n".join(lines))
    print(f"\n🚀  Published: {filepath}")
    return {"published_filepath": filepath}

# ─── Guardrails ───────────────────────────────────────────────────────────────
def guardrail_or_stop(state) -> Literal["approved", "retry_editor", "force_end"]:
    if state.get("retry_count", 0) >= MAX_RETRIES:
        return "force_end"
    article = state.get("final_article")
    if not article:
        return "retry_editor"
    words = (len(article.introduction.split()) +
             sum(len(s.content.split()) for s in article.sections) +
             len(article.conclusion.split()))
    if not (500 <= words <= 1500):
        state["validation_feedback"] = f"Word count {words} outside 500–1500 range."
        return "retry_editor"
    if len(article.sections) < 4:
        state["validation_feedback"] = f"Only {len(article.sections)} sections; need at least 4."
        return "retry_editor"
    return "approved"

def route_after_outline_review(state) -> Literal["writer", "researcher"]:
    return "researcher" if state.get("human_outline_feedback") else "writer"

def route_after_article_review(state) -> Literal["approved", "editor"]:
    return "approved" if state.get("human_approved") else "editor"

# ─── Build Graph ──────────────────────────────────────────────────────────────
memory = MemorySaver()
graph  = StateGraph(AgentState)

graph.add_node("researcher",           researcher_node)
graph.add_node("human_review_outline", human_review_outline_node)
graph.add_node("writer",               writer_node)
graph.add_node("editor",               editor_node)
graph.add_node("human_review_article", human_review_article_node)
graph.add_node("publish",              publish_node)

graph.set_entry_point("researcher")
graph.add_edge("researcher", "human_review_outline")
graph.add_conditional_edges("human_review_outline", route_after_outline_review,
                            {"writer": "writer", "researcher": "researcher"})
graph.add_edge("writer", "editor")
graph.add_conditional_edges("editor", guardrail_or_stop,
                            {"approved": "human_review_article",
                             "retry_editor": "editor", "force_end": END})
graph.add_conditional_edges("human_review_article", route_after_article_review,
                            {"approved": "publish", "editor": "editor"})
graph.add_edge("publish", END)

app = graph.compile(checkpointer=memory)

# ─── Run ──────────────────────────────────────────────────────────────────────
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

initial_state = {
    "topic": "The Future of Space Exploration",
    "raw_research": "", "research_outline": None,
    "draft_article": None, "final_article": None,
    "validation_feedback": "", "retry_count": 0,
    "human_outline_feedback": "", "human_article_feedback": "",
    "human_approved": False, "published_filepath": ""
}

for event in app.stream(initial_state, config=config, stream_mode="updates"):
    node = list(event.keys())[0]
    if node != "__interrupt__":
        print(f"→ {node}")
    else:
        msg = event["__interrupt__"][0].value
        response = input(f"\n⏸  {msg}\nYour response: ").strip()
        for _ in app.stream(Command(resume=response), config=config, stream_mode="updates"):
            pass




In [ ]:
# Visualize the full graph (requires graphviz)
from IPython.display import Image
Image(app.get_graph(xray=True).draw_mermaid_png())